# Module 2 · Lesson 01: Zero-Shot Prompting

**Zero-shot prompting** means asking the model to perform a task *without any examples*.
The model relies entirely on its training data and your instructions.

## What you will learn
1. Effective zero-shot prompt patterns
2. Output **format specification**
3. **Role/persona** assignment
4. Zero-shot **classification**
5. **Structured output** (JSON) extraction
6. Using **constraints** to control output

In [5]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
if client:
    print("Client is ready")
 

Client is ready


In [6]:
def ask(prompt, system=None, temperature = 0.7, max_tokens=1200):
    msgs = []
    if system:
        msgs.append({"role":"system", "content":system})
    msgs.append({"role":"user", "content":prompt})
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=msgs,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return r.choices[0].message.content

---
## 1. Simple Zero-Shot

The simplest form — just ask directly:

In [7]:
result = ask("What is the capital of France?")
display(Markdown(f"**Q**: What is the capital of France?\n\n**A**: {result}"))

**Q**: What is the capital of France?

**A**: The capital of France is Paris.

---
## 2. Format Specification

Tell the model **exactly** what format you want:

In [9]:
prompt = """Extract the email address from this text and return ONLY the email, nothing else:

Hi, you can reach me at john.doe@example.com for more information.
"""
result = ask(prompt, max_tokens=50)
display(Markdown(f"**Extracted:** `{result.strip()}`"))

**Extracted:** `john.doe@example.com`

---
## 3. Role / Persona Assignment

The system prompt sets *who* the model should be:

In [ ]:
result = ask(
    prompt="Explain what an API is to a non-technical person.",
    system="You are a professiona technical writer who explains complex concenpts simply"
)

display(Markdown(f"### Technical Writer\n\n{result}"))

### Technical Writer

An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that applications can use to request and exchange information. APIs are essential for enabling the integration of different systems and services, allowing them to work together seamlessly.

Here are some key components of APIs:

1. **Endpoints**: These are specific URLs or URIs where the API can be accessed. Each endpoint corresponds to a different function or resource.

2. **Requests and Responses**: An API typically works through requests made by a client (such as a web application) to a server (where the API is hosted). The request can include parameters and data, and the server responds with the requested information or the result of an action.

3. **Methods**: Common HTTP methods used in APIs include:
   - **GET**: Retrieve data from the server.
   - **POST**: Send data to the server to create a new resource.
   - **PUT**: Update an existing resource on the server.
   - **DELETE**: Remove a resource from the server.

4. **Data Formats**: APIs often use standardized data formats such as JSON (JavaScript Object Notation) or XML (eXtensible Markup Language) to structure the data being exchanged.

5. **Authentication and Security**: Many APIs require authentication to ensure that only authorized users can access certain resources. This can involve API keys, OAuth tokens, or other authentication methods.

APIs are widely used in web development, mobile applications, cloud services, and many other areas, enabling developers to build more complex applications by leveraging existing services and functionalities. They play a crucial role in modern software development, allowing for modular, reusable, and scalable code.

---
## 4. Zero-Shot Classification

LLMs are excellent **zero-shot classifiers**. No training data needed!

In [18]:
texts = [
    "I absolutely love this product! Best purchase ever",
    "The shipping was delayed and the item arrived damaged",
    "It's okay, nothing special but does the job"
]

print(f"{'Text':<55} {'Sentiment'}")
print("-" * 101)

for text in texts:
    prompt = f"Classify the sentiment as exactly one word: positive, negative or neural\n\nText: {text}\n\nClassification:"
    sentiment = ask(prompt, temperature=0, max_tokens=10).strip().lower()
    emoji = {"positive": "🟢", "negative": "🔴", "neutral": "🟡"}.get(sentiment, "⚪")
    print(f"{text[:52]+'...':<55} {emoji} {sentiment}")

Text                                                    Sentiment
-----------------------------------------------------------------------------------------------------
I absolutely love this product! Best purchase ever...   🟢 positive
The shipping was delayed and the item arrived damage... 🔴 negative
It's okay, nothing special but does the job...          🟡 neutral


# Role - Context - Structure

In [20]:
result = ask(
    prompt="Explain what an API is to 10 years old child. Do it in 3 short sentences. Put them in ordered bullets",
    system="You are my school teacher who loves analogies and explain everything in easy way."
)

display(Markdown(f"### Technical Writer:\n\n{result}"))

### Technical Writer:

Sure! Here’s an easy way to understand what an API is:

1. Imagine your favorite restaurant: you tell the waiter what you want to eat, and they bring it to you from the kitchen.  
2. An API (Application Programming Interface) is like that waiter, helping two different computer programs talk to each other and share information.  
3. So, when you use an app on your tablet and it gets data from the internet, it's like the app is ordering food from the kitchen through the API!  

> 💡 Use `temperature=0` for classification to get deterministic, consistent results.

---
## 5. Structured Output (JSON)

In [26]:
import json

prompt = """Extract information from this text and return as valid JSON:

Text: "Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss te Q1 budget report."

Return JSON fields: attendee, date, time, topic

Do NOT wrap the output in markdown code fences or any other formating.
Return ONLY the raw JSON object, nothing else.

JSON:"""

result = ask(prompt, temperature=0)
try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [27]:
response_json = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user", "content": prompt}],
    response_format={"type":"json_object"}, # force valid JSON output
    temperature=0
)

new_res = response_json.choices[0].message.content

try:
    parsed = json.loads(new_res)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {new_res}")
    print("Not valid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [28]:
import json

response_json = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user", "content":"Return a short greeting and a lucky number"}],
    response_format={
        "type":"json_schema",
        "json_schema": {
            "name": "greeting_response",
            "schema": {
                "type": "object",
                "properties": {
                    "greeting": {"type": "string"},
                    "luckyNumber": {"type": "integer"}
                },
                "required": ["greeting", "luckyNumber"],
                "additionalProperties":False
            },
            "strict":True
        }
    },
    temperature=0
)

new_res = response_json.choices[0].message.content
parsed = json.loads(new_res)

print(json.dumps(parsed, indent=2))

{
  "greeting": "Hello! Wishing you a wonderful day!",
  "luckyNumber": 7
}


In [29]:
import json
 
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Describe an API endpoint for user login."}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "api_endpoint",
            "schema": {
                "type": "object",
                "properties": {
                    "endpoint": {"type": "string"},
                    "method": {
                        "type": "string",
                        "enum": ["GET", "POST", "PUT", "DELETE"]
                    },
                    "request_body": {
                        "type": "object",
                        "properties": {
                            "email": {"type": "string"},
                            "password": {"type": "string"}
                        },
                        "required": ["email", "password"],
                        "additionalProperties": False
                    },
                    "response": {
                        "type": "object",
                        "properties": {
                            "token": {"type": "string"},
                            "expires_in": {"type": "integer"}
                        },
                        "required": ["token", "expires_in"],
                        "additionalProperties": False
                    }
                },
                "required": ["endpoint", "method", "request_body", "response"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    temperature=0
)
 
parsed = json.loads(response.choices[0].message.content)
print(json.dumps(parsed, indent=2))

{
  "endpoint": "/api/login",
  "method": "POST",
  "request_body": {
    "email": "user@example.com",
    "password": "yourpassword"
  },
  "response": {
    "token": "abc123xyz",
    "expires_in": 3600
  }
}


---
## 6. Constraints

Adding explicit **constraints** controls length, format, and content:

In [32]:
prompt = """Write a product description for a wireless mouse.

Constraints:
- Maximum 50 words
- Include at least one benefit
- Do not mention price
- End with a call to action

Description:"""

result = ask(prompt)
word_count = len(result.split())

display(Markdown(f"> {result}"))
print(f"word_count: {word_count}")

> Elevate your productivity with our sleek wireless mouse. Experience seamless navigation and unparalleled comfort, perfect for long hours of work or gaming. Enjoy the freedom of movement without tangled wires. Upgrade your workspace today and discover the difference—grab yours now!

word_count: 40


### Multi-dimentional constraints (style - structure - semantics)

In [35]:
prompt = """
Write a product description for wireless mouse.

Constaints:
- 40-60 words
- Exactly 2 sentences
- First sentence: describe features
- Second sentece: emphasize a user benefit
- Include exactly 1 emoji
- Must contain the word "precision"
- Do NOT use passive voice
- End with a call to action

Return ONLY the description.
"""

result = ask(prompt)
display(Markdown(f" > {result}"))
print(f"Word count: {len(result.split())}")

 > Experience seamless navigation with our wireless mouse, featuring adjustable DPI settings for precision, ergonomic design for comfort, and a long-lasting battery. Elevate your productivity and enjoy effortless control—grab yours today! 🖱️

Word count: 31


---
## 7. Prompt Gallery: Real-World System Prompts

Let's study system prompts from **real community projects**. Each uses a different technique
to get reliable, high-quality outputs.

| Source | Key Technique |
|--------|---------------|
| Gmail Summarizer | Structured rules + labels |
| Budget Travel Agent | Role + constraint + format |
| Biomedical Summariser | Audience awareness |
| Code Explainer | Section structure |

In [36]:
# ── Prompt Gallery: 4 real-world system prompts ─────────
 
gallery = {
    "Gmail Summarizer": {
        "prompt": """You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.""",
        "technique": "Structured rules with labels and constraints",
    },
    "Budget Travel Agent": {
        "prompt": """You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.""",
        "technique": "Role + numbered constraints + format (markdown)",
    },
    "Biomedical Summariser": {
        "prompt": """Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.""",
        "technique": "Audience awareness + structure + tone",
    },
    "Code Explainer": {
        "prompt": """You explain code to developers. Structure your response as:
1) Direct Answer (1-2 sentences)
2) Explanation (why it works)
3) Example (working code snippet)
4) Common Pitfalls (what to avoid)
5) Next Steps (what to learn next)""",
        "technique": "Section-structured output format",
    }
}
 
# Display each prompt with analysis
for name, info in gallery.items():
    print(f"\n{'=' * 60}")
    print(f"  {name}")
    print(f"  Technique: {info['technique']}")
    print(f"{'=' * 60}")
    print(info['prompt'])


  Gmail Summarizer
  Technique: Structured rules with labels and constraints
You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.

  Budget Travel Agent
  Technique: Role + numbered constraints + format (markdown)
You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.

  Biomedical Summariser
  Technique: Audience awareness + structure + tone
Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.

  Code Explainer
  Technique: Section-structured output format
You explain code to developers. 

In [37]:
travel_result = ask(
    prompt="I'm visiting Lisbon, Portugal for 3 days on a tight budget.",
    system=gallery["Budget Travel Agent"]["prompt"]
)

display(Markdown(f"### Budget Travel Agent Response\n\n{travel_result}"))

### Budget Travel Agent Response

# Budget Travel Guide to Lisbon, Portugal

## Top 5 FREE Attractions
1. **Belém Tower (Torre de Belém)**: While the inside requires a ticket, you can explore the area and enjoy the views of the tower from the outside for free.
2. **Alfama District**: Wander through the narrow streets of Lisbon's oldest neighborhood and soak in the atmosphere, colorful buildings, and stunning viewpoints.
3. **Miradouro de Santa Catarina**: Enjoy panoramic views of the city and the Tagus River from this popular viewpoint, perfect for sunset watching.
4. **Lisbon Street Art**: Take a self-guided tour of the city’s vibrant street art scene, particularly in neighborhoods like Bairro Alto and Graça.
5. **Parque Eduardo VII**: This large park offers beautiful gardens and a great view of the city. It's a perfect spot for a picnic or a leisurely walk.

## Budget Restaurants (Under $15/Meal)
1. **Time Out Market**: While some stalls can be pricey, you can find budget-friendly options like sandwiches and tapas at various vendors.
2. **Cervejaria Ramiro**: Famous for its seafood, you can find affordable dishes like garlic shrimp and crab. Look for smaller portions to keep costs down.
3. **Casa dos Passarinhos**: Known for its delicious grilled meats and vegetarian options, this local eatery offers hearty meals at very reasonable prices.

## Money-Saving Local Tip
**Use Public Transport**: Purchase a 24-hour public transport pass, which includes access to trams, buses, and the metro. It’s a cost-effective way to get around the city and explore various neighborhoods without breaking the bank.

In [39]:
gmail_result = ask(
    """Here's an email thread:
From: marketing@company.com
Subject: Re: Q2 Campaign Launch — Asset Review Needed
Body: Hi team, please review the attached creatives for the Q2 social campaign.
We need approvals by Friday. The campaign landing page is at https://company.com/q2launch.
Let me know if any changes are needed. Thanks!""",
    system=gallery["Gmail Summarizer"]["prompt"]
)
display(Markdown(f"### Gmail Summarizer Response\n\n{gmail_result}"))
print("\nNotice the consistent labeling and structured bullet format!")

### Gmail Summarizer Response

1. Subject line: Q2 Campaign Launch — Asset Review Needed  
   Label: ACTION_REQUIRED  
   Summary: The marketing team requests reviews and approvals of the attached creatives for the Q2 social campaign by Friday. The campaign landing page link is included for reference.  
   Has link: yes


Notice the consistent labeling and structured bullet format!


In [40]:
bio_result = ask(
    """A 2024 study published in The Lancet examined the efficacy of a novel mRNA-based
therapeutic vaccine for stage III melanoma. The randomized, double-blind trial enrolled
340 patients across 22 clinical sites. Results showed a 44% reduction in recurrence risk
(HR 0.56, 95% CI 0.40–0.78, p=0.0007) over a 24-month follow-up. Common adverse effects
included fatigue (32%) and injection-site reactions (28%). The authors noted the relatively
short follow-up period and homogeneous population (predominantly Caucasian, median age 58)
as key limitations, and called for larger Phase III trials with diverse cohorts.""",
    system=gallery["Biomedical Summariser"]["prompt"]
)
display(Markdown(f"### Biomedical Summariser Response\n\n{bio_result}"))
print("\nNotice how it highlights methodology, limitations, and stays accessible!")

### Biomedical Summariser Response

**Study Summary: Efficacy of an mRNA-based Therapeutic Vaccine for Stage III Melanoma**  
*Published in The Lancet, 2024*

- **Key Findings:**
  - The novel mRNA-based therapeutic vaccine showed a **44% reduction in recurrence risk** for stage III melanoma patients.
  - Hazard Ratio (HR) reported was **0.56** (95% Confidence Interval [CI] 0.40–0.78, p=0.0007) over a **24-month follow-up** period.

- **Methodology:**
  - The study was a **randomized, double-blind trial**.
  - **Sample Size:** 340 patients.
  - Enrolled patients were from **22 clinical sites**.

- **Adverse Effects:**
  - Most common side effects included:
    - **Fatigue:** 32%
    - **Injection-site reactions:** 28%

- **Limitations:**
  - The follow-up period was relatively short, limiting long-term efficacy assessment.
  - The study population was **homogeneous**, predominantly Caucasian with a median age of 58, which may affect the generalizability of results.

- **Future Directions:**
  - The authors recommend conducting **larger Phase III trials** that include **more diverse cohorts** to validate findings and explore efficacy across different demographics.


Notice how it highlights methodology, limitations, and stays accessible!


In [41]:
code_result = ask(
    """Explain this Python code:
result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}""",
    system=gallery["Code Explainer"]["prompt"]
)
display(Markdown(f"### Code Explainer Response\n\n{code_result}"))
print("\nNotice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!")

### Code Explainer Response

1) **Direct Answer:** This Python code creates a dictionary called `result` that contains the top 5 key-value pairs from the `data` dictionary, sorted by their values in descending order.

2) **Explanation:** The code uses a dictionary comprehension to iterate over the items of `data` after sorting them by their values (item[1]) in descending order (reverse=True). The slicing `[:5]` ensures that only the top 5 items are taken from the sorted list.

3) **Example:**
   ```python
   data = {
       'apple': 10,
       'banana': 20,
       'cherry': 15,
       'date': 5,
       'fig': 25,
       'grape': 30,
       'kiwi': 7
   }

   result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}
   print(result)  # Output: {'grape': 30, 'fig': 25, 'banana': 20, 'cherry': 15, 'apple': 10}
   ```

4) **Common Pitfalls:** A common mistake is not handling cases where `data` has fewer than 5 items, which can lead to unexpected results. It's also important to ensure that all values in `data` are comparable; otherwise, a `TypeError` may occur if the values are of different types.

5) **Next Steps:** To deepen your understanding, explore other sorting techniques in Python, learn about dictionary methods, and practice using list comprehensions and lambda functions for more complex data manipulations.


Notice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!


> **Exercise:** Pick a business domain you're interested in (e.g., fitness coaching,
> legal review, recipe generation) and write your own system prompt following the
> patterns above. Test it with 3 different user queries.

---
## Key Takeaways 📝

| Technique | When to Use |
|-----------|------------|
| **Direct question** | Simple factual queries |
| **Format specification** | When you need specific output format |
| **Role assignment** | To control tone, expertise, style |
| **Classification** | Categorizing text (use temp=0) |
| **JSON extraction** | Structured data from unstructured text |
| **Constraints** | Controlling length, style, content boundaries |
| **Prompt gallery** | Study real prompts to develop critical analysis skills |

### Zero-Shot Tips
1. Be **specific** about what you want
2. Specify the **output format** explicitly
3. Use **roles/personas** to guide behaviour
4. Add **negative constraints** (what NOT to do)
5. Use `temperature=0` for deterministic tasks

---
**Next:** `02_few_shot_examples.ipynb` — Improve results by providing examples